# NEF Anomaly Detection
## Flagging Outliers in South African Development Finance
### Isolation Forest · Public Accountability Analysis

---

> *Not all outliers are failures. Some companies deliver extraordinary value for public money.
> Others absorb millions and create almost nothing. Isolation Forest identifies both —
> without human judgment about which companies to flag.
> The algorithm decides based on the data alone.
> What it finds is a question for policymakers to answer.*

---

**Method:** Isolation Forest (sklearn) — an unsupervised ML algorithm that detects anomalies
by isolating observations that are statistically unusual across multiple dimensions simultaneously.
Unlike Z-score or IQR methods, Isolation Forest captures multivariate anomalies —
a company that is unusual across disbursement size, jobs created, AND province simultaneously.

**Two anomaly labels:**
- 🔴 **Red Flag** — anomalously high cost-per-job relative to disbursement and province
- 🟢 **Positive Outlier** — anomalously low cost-per-job, exceptional job creation efficiency

**Features:**
- `log_disbursed` — log of total disbursed amount
- `log_cpj` — log of cost per job
- `log_jobs` — log of jobs created
- `bracket_ord` — deal size bracket (ordinal 1–6)
- `has_grant` — binary grant indicator
- Province dummies — 9 SA provinces

**Contamination parameter:** 10% — meaning the model expects roughly 10% of the dataset
to be anomalous. This is consistent with what Project 3 found: the top and bottom deciles
of cost-per-job efficiency are where the most extreme outcomes live.

**Data note:** 392 records — 17 real company anchors from PQ705,
375 synthesised from provincial aggregates. Anomaly findings for real companies
are the primary analytical focus.

**Audience:** Data scientists, policy analysts, development finance professionals

---
## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0f1117', 'axes.facecolor': '#1a1d2e',
    'axes.edgecolor': '#444',     'axes.labelcolor': '#ccc',
    'xtick.color': '#aaa',        'ytick.color': '#aaa',
    'text.color': '#eee',         'grid.color': '#333',
    'grid.linestyle': '--',       'grid.alpha': 0.5,
    'font.family': 'sans-serif',  'figure.dpi': 120,
})
GOLD='#f0a500'; TEAL='#00c9a7'; RED='#e05c5c'; BLUE='#4e8df5'
GREY='#6b7280'; GREEN='#34d399'; PURPLE='#a78bfa'

np.random.seed(42)
print('Setup complete ✓')

---
## 1. Data Construction
### Same dataset as Project 2 — with company names preserved for real records

In [ ]:
nef_provincial = pd.DataFrame({
    'province':      ['Gauteng','KwaZulu-Natal','Northern Cape','Western Cape',
                      'Eastern Cape','Mpumalanga','Limpopo','Free State','North West'],
    'deals':         [135, 122, 14, 31, 23, 18, 19, 10, 20],
    'avg_per_deal':  [7713104, 7955172, 45814286, 8771129,
                      8843478, 8335667, 6998053, 10923900, 3782800],
    'avg_jobs_deal': [104.7, 69.2, 23.3, 44.0, 82.8, 61.0, 91.6, 188.1, 38.2],
    'cost_per_job':  [73650, 114992, 1967485, 199490,
                      106828, 136650, 76372, 58075, 99156],
})

real_companies = pd.DataFrame([
    {'company':'Khatu Industrial & Chemical',  'province':'Northern Cape','disbursed':534000000,'jobs':26, 'has_grant':0},
    {'company':'CK Mafutha (Pty) Ltd',         'province':'Western Cape', 'disbursed':77200000, 'jobs':4,  'has_grant':0},
    {'company':'Devland Gardens (Pty) Ltd',    'province':'Gauteng',      'disbursed':49000000, 'jobs':16, 'has_grant':0},
    {'company':"Africa's Best 350 (Pty) Ltd",  'province':'Eastern Cape', 'disbursed':44500000, 'jobs':442,'has_grant':0},
    {'company':'Mandini Group',                'province':'Gauteng',      'disbursed':44400000, 'jobs':142,'has_grant':0},
    {'company':'Hayett Investments (Pty) Ltd', 'province':'KwaZulu-Natal','disbursed':43600000, 'jobs':230,'has_grant':0},
    {'company':'Dandelton Investments',        'province':'KwaZulu-Natal','disbursed':43300000, 'jobs':96, 'has_grant':0},
    {'company':'Salim Munshi Family Trust',    'province':'KwaZulu-Natal','disbursed':38700000, 'jobs':613,'has_grant':0},
    {'company':'Greyline Holdings',            'province':'Gauteng',      'disbursed':38200000, 'jobs':258,'has_grant':0},
    {'company':'Suntrans CC',                  'province':'KwaZulu-Natal','disbursed':38100000, 'jobs':74, 'has_grant':0},
    {'company':'Umnotho Maize (Pty) Ltd',      'province':'Gauteng',      'disbursed':9000000,  'jobs':2352,'has_grant':0},
    {'company':'Icebolethu Burial Services',   'province':'KwaZulu-Natal','disbursed':19100000, 'jobs':1843,'has_grant':0},
    {'company':'Tshellaine Holdings',          'province':'Gauteng',      'disbursed':37800000, 'jobs':1664,'has_grant':0},
    {'company':'Lebowakgomo Poultry (Pty) Ltd','province':'Limpopo',      'disbursed':1600000,  'jobs':887,'has_grant':0},
    {'company':'KPML Group (Pty) Ltd',         'province':'Gauteng',      'disbursed':2000000,  'jobs':805,'has_grant':0},
    {'company':'Bibi Cash & Carry Superstore', 'province':'Free State',   'disbursed':27700000, 'jobs':785,'has_grant':0},
    {'company':'Ubettina Wethu Company',       'province':'Gauteng',      'disbursed':5000000,  'jobs':593,'has_grant':0},
])
real_companies['cost_per_job']  = real_companies['disbursed'] / real_companies['jobs']
real_companies['log_disbursed'] = np.log(real_companies['disbursed'])
real_companies['log_cpj']       = np.log(real_companies['cost_per_job'])
real_companies['log_jobs']      = np.log(real_companies['jobs'])
real_companies['bracket_ord']   = real_companies['disbursed'].apply(
    lambda x: 1 if x<1e6 else 2 if x<5e6 else 3 if x<10e6
              else 4 if x<25e6 else 5 if x<50e6 else 6)
real_companies['is_real'] = True

records = []
for _, p in nef_provincial.iterrows():
    for _ in range(p['deals']):
        disbursed = max(100000, np.random.lognormal(np.log(p['avg_per_deal']), 0.8))
        jobs      = max(1, int(np.random.lognormal(np.log(max(1,p['avg_jobs_deal'])), 0.7)))
        cpj       = disbursed / jobs
        bord = (1 if disbursed<1e6 else 2 if disbursed<5e6 else 3 if disbursed<10e6
                else 4 if disbursed<25e6 else 5 if disbursed<50e6 else 6)
        records.append({
            'company': f'Company_{len(records)+1}',
            'province': p['province'], 'disbursed': disbursed, 'jobs': jobs,
            'cost_per_job': cpj, 'bracket_ord': bord,
            'has_grant': int(np.random.random() < 0.196),
            'log_disbursed': np.log(disbursed), 'log_cpj': np.log(cpj),
            'log_jobs': np.log(jobs), 'is_real': False,
        })

df_synth = pd.DataFrame(records)
df = pd.concat([df_synth.iloc[:-17],
                real_companies.reindex(columns=df_synth.columns, fill_value=0)],
               ignore_index=True)

print(f'Dataset: {len(df)} records ({df["is_real"].sum()} real, {(~df["is_real"]).sum()} synthetic)')
print(f'Cost per job range: R{df["cost_per_job"].min():,.0f} – R{df["cost_per_job"].max():,.0f}')
print(f'Jobs range: {int(df["jobs"].min())} – {int(df["jobs"].max())}')

---
## 2. Exploratory Analysis
### Understanding the distributions before anomaly detection

In [ ]:
# ── Disbursed vs Jobs scatter (raw — showing where outliers visually sit) ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw scatter
axes[0].scatter(df['disbursed']/1e6, df['jobs'],
                alpha=0.35, s=20, color=GREY, label='Synthetic')
real = df[df['is_real']==True]
axes[0].scatter(real['disbursed']/1e6, real['jobs'],
                alpha=0.9, s=70, color=GOLD,
                edgecolors='white', lw=0.8, label='Real (PQ705)', zorder=5)

# Annotate extreme real companies
for _, row in real.iterrows():
    if row['disbursed'] > 100e6 or row['jobs'] > 1500 or\
       (row['disbursed'] > 40e6 and row['jobs'] < 30):
        axes[0].annotate(
            row['company'].split('(')[0].strip()[:20],
            (row['disbursed']/1e6, row['jobs']),
            textcoords='offset points', xytext=(6, 4),
            fontsize=7.5, color=RED
        )

axes[0].set_xlabel('Disbursed (R millions)')
axes[0].set_ylabel('Jobs Created')
axes[0].set_title('Disbursed vs Jobs Created\n(Gold = real companies)', fontweight='bold', color='white')
axes[0].legend(fontsize=9)
axes[0].grid(zorder=0)

# Log-log scatter — linearises the relationship
axes[1].scatter(df['log_disbursed'], df['log_jobs'],
                alpha=0.35, s=20, color=GREY)
axes[1].scatter(real['log_disbursed'], real['log_jobs'],
                alpha=0.9, s=70, color=GOLD,
                edgecolors='white', lw=0.8, zorder=5)
axes[1].set_xlabel('log(Disbursed)')
axes[1].set_ylabel('log(Jobs)')
axes[1].set_title('Log-Log Space\n(Used by Isolation Forest)', fontweight='bold', color='white')
axes[1].grid(zorder=0)

plt.suptitle('NEF: Disbursement vs Job Creation — Raw and Log-Transformed',
             fontsize=12, fontweight='bold', color='white', y=1.02)
plt.tight_layout()
plt.savefig('chart_anomaly_eda.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('Two clusters are already visible in log-log space.')
print('Upper left = high jobs, low disbursement (efficient).')
print('Lower right = low jobs, high disbursement (inefficient).')

---
## 3. Feature Engineering & Isolation Forest
### Why Isolation Forest?

Isolation Forest works by randomly partitioning the feature space and counting how many
splits it takes to isolate each observation. Anomalies are isolated quickly
(fewer splits needed) because they sit far from the dense cluster of normal observations.

This is ideal for this dataset because:
- We are not looking for anomalies on a single axis (just cost-per-job)
- We want to capture companies that are unusual across *multiple dimensions simultaneously*
- We have no labelled anomalies to train on (unsupervised is required)
- The dataset is relatively small (392 records) — Isolation Forest is robust at this scale

In [ ]:
# ── Feature matrix ────────────────────────────────────────────────────────
province_dummies = pd.get_dummies(df['province'], prefix='prov')
features = pd.concat([
    df[['log_disbursed','log_cpj','log_jobs','bracket_ord','has_grant']],
    province_dummies
], axis=1)

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(features)

print('Feature matrix shape:', features.shape)
print('Features:', features.columns.tolist())

# ── Fit Isolation Forest ──────────────────────────────────────────────────
iso = IsolationForest(
    contamination=0.10,
    n_estimators=200,
    max_samples='auto',
    random_state=42
)
df['anomaly_score']  = iso.fit_predict(X_scaled)   # 1=normal, -1=anomaly
df['anomaly_raw']    = iso.score_samples(X_scaled)  # lower = more anomalous

# ── Direction labelling ───────────────────────────────────────────────────
cpj_median = df['log_cpj'].median()
df['anomaly_label'] = 'Normal'
df.loc[(df['anomaly_score']==-1) & (df['log_cpj'] <  cpj_median), 'anomaly_label'] = 'Positive Outlier'
df.loc[(df['anomaly_score']==-1) & (df['log_cpj'] >= cpj_median), 'anomaly_label'] = 'Red Flag'

print(f'\nTotal anomalies: {(df["anomaly_score"]==-1).sum()} ({(df["anomaly_score"]==-1).mean()*100:.1f}%)')
print(f'Red Flags:        {(df["anomaly_label"]=="Red Flag").sum()}')
print(f'Positive Outliers:{(df["anomaly_label"]=="Positive Outlier").sum()}')
print(f'Normal:           {(df["anomaly_label"]=="Normal").sum()}')
print(f'\nAnomaly score range: {df["anomaly_raw"].min():.4f} to {df["anomaly_raw"].max():.4f}')

---
## 4. Anomaly Score Distribution

In [ ]:
threshold = df['anomaly_raw'].quantile(0.10)

fig, ax = plt.subplots(figsize=(12, 5))

normal   = df[df['anomaly_label']=='Normal']['anomaly_raw']
red_flag = df[df['anomaly_label']=='Red Flag']['anomaly_raw']
positive = df[df['anomaly_label']=='Positive Outlier']['anomaly_raw']

ax.hist(normal,   bins=35, color=GREY,  alpha=0.6, label='Normal',           zorder=2)
ax.hist(red_flag, bins=15, color=RED,   alpha=0.85, label='Red Flag',         zorder=3)
ax.hist(positive, bins=15, color=GREEN, alpha=0.85, label='Positive Outlier', zorder=3)

ax.axvline(threshold, color='white', linestyle='--', lw=1.5,
           label=f'Anomaly threshold ({threshold:.3f})')

# Mark real company positions
real_anom = df[(df['is_real']==True) & (df['anomaly_score']==-1)]
for _, row in real_anom.iterrows():
    color = RED if row['anomaly_label']=='Red Flag' else GREEN
    ax.axvline(row['anomaly_raw'], color=color, lw=1.0, alpha=0.7, linestyle=':')
    ax.text(row['anomaly_raw'], ax.get_ylim()[1]*0.85 if ax.get_ylim()[1] > 0 else 5,
            row['company'].split('(')[0].strip()[:15],
            rotation=90, fontsize=6.5, color=color, ha='center')

ax.set_xlabel('Anomaly Score (lower = more anomalous)')
ax.set_ylabel('Count')
ax.set_title('Isolation Forest Anomaly Score Distribution\n'
             'Dotted lines = real flagged companies',
             fontweight='bold', color='white', pad=12)
ax.legend(fontsize=9)
ax.grid(zorder=0)
plt.tight_layout()
plt.savefig('chart_anomaly_scores.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()

---
## 5. Anomaly Map — Disbursed vs Jobs
### Visualising where flagged companies sit in feature space

In [ ]:
color_map = {'Normal': GREY, 'Red Flag': RED, 'Positive Outlier': GREEN}
size_map  = {'Normal': 18,   'Red Flag': 60,  'Positive Outlier': 60}
alpha_map = {'Normal': 0.25, 'Red Flag': 0.9, 'Positive Outlier': 0.9}

fig, ax = plt.subplots(figsize=(12, 7))

for label in ['Normal','Red Flag','Positive Outlier']:
    subset = df[df['anomaly_label']==label]
    ax.scatter(
        subset['log_disbursed'], subset['log_jobs'],
        c=color_map[label], s=size_map[label],
        alpha=alpha_map[label], label=label,
        edgecolors='white' if label != 'Normal' else 'none',
        linewidths=0.5, zorder=3 if label != 'Normal' else 2
    )

# Annotate real flagged companies
for _, row in df[(df['is_real']==True) & (df['anomaly_score']==-1)].iterrows():
    ax.annotate(
        row['company'].split('(')[0].strip()[:22],
        (row['log_disbursed'], row['log_jobs']),
        textcoords='offset points',
        xytext=(8, 4),
        fontsize=8,
        color=RED if row['anomaly_label']=='Red Flag' else GREEN,
        fontweight='bold'
    )

ax.set_xlabel('log(Disbursed Amount)')
ax.set_ylabel('log(Jobs Created)')
ax.set_title('Anomaly Map: NEF Funded Companies\n'
             '🔴 Red Flag = poor value for money  ·  🟢 Positive Outlier = exceptional efficiency',
             fontweight='bold', color='white', pad=12)
ax.legend(fontsize=10, markerscale=1.2)
ax.grid(zorder=0)
plt.tight_layout()
plt.savefig('chart_anomaly_map.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('Red flags cluster bottom-right: high disbursement, few jobs.')
print('Positive outliers cluster top-left: low disbursement, many jobs.')

---
## 6. The Named Outliers
### Real companies flagged by the algorithm

In [ ]:
real_flagged = df[(df['is_real']==True) & (df['anomaly_score']==-1)].copy()
real_flagged = real_flagged.sort_values('anomaly_raw')

print('='*70)
print('RED FLAGS — Public money, poor job creation outcomes')
print('='*70)
red_real = real_flagged[real_flagged['anomaly_label']=='Red Flag']
for _, row in red_real.iterrows():
    print(f"\n  Company:      {row['company']}")
    print(f"  Province:     {row['province']}")
    print(f"  Disbursed:    R{row['disbursed']:,.0f}")
    print(f"  Jobs:         {int(row['jobs'])}")
    print(f"  Cost/job:     R{row['cost_per_job']:,.0f}")
    print(f"  Anomaly score:{row['anomaly_raw']:.4f} (lower = more anomalous)")

print()
print('='*70)
print('POSITIVE OUTLIERS — Exceptional job creation per rand')
print('='*70)
pos_real = real_flagged[real_flagged['anomaly_label']=='Positive Outlier']
for _, row in pos_real.iterrows():
    print(f"\n  Company:      {row['company']}")
    print(f"  Province:     {row['province']}")
    print(f"  Disbursed:    R{row['disbursed']:,.0f}")
    print(f"  Jobs:         {int(row['jobs'])}")
    print(f"  Cost/job:     R{row['cost_per_job']:,.0f}")
    print(f"  Anomaly score:{row['anomaly_raw']:.4f}")

In [ ]:
# ── Red flag concentration by province ───────────────────────────────────
prov_summary = df.groupby('province').agg(
    total_deals    = ('company', 'count'),
    red_flags      = ('anomaly_label', lambda x: (x=='Red Flag').sum()),
    pos_outliers   = ('anomaly_label', lambda x: (x=='Positive Outlier').sum()),
    total_anomalies= ('anomaly_score', lambda x: (x==-1).sum()),
).reset_index()
prov_summary['red_flag_rate'] = prov_summary['red_flags'] / prov_summary['total_deals'] * 100
prov_summary['pos_rate']      = prov_summary['pos_outliers'] / prov_summary['total_deals'] * 100
prov_summary = prov_summary.sort_values('red_flag_rate', ascending=False)

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(prov_summary))
w = 0.38
ax.bar(x - w/2, prov_summary['red_flag_rate'],  w, color=RED,   alpha=0.85, label='Red Flag Rate (%)')
ax.bar(x + w/2, prov_summary['pos_rate'],       w, color=GREEN, alpha=0.85, label='Positive Outlier Rate (%)')
ax.set_xticks(x)
ax.set_xticklabels(prov_summary['province'], rotation=25, ha='right', fontsize=9)
ax.set_ylabel('% of Deals in Province')
ax.set_title('Anomaly Rates by Province\n'
             'Which provinces have the highest concentration of red flags?',
             fontweight='bold', color='white', pad=12)
ax.legend(fontsize=10)
ax.grid(axis='y', zorder=0)
plt.tight_layout()
plt.savefig('chart_anomaly_provinces.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print(prov_summary[['province','total_deals','red_flags','pos_outliers',
                     'red_flag_rate','pos_rate']].to_string(index=False))

---
## 7. Sensitivity Analysis
### How stable are the findings across contamination rates?

In [ ]:
# ── Test contamination = 0.05, 0.10, 0.15, 0.20 ───────────────────────────
contamination_rates = [0.05, 0.10, 0.15, 0.20]
real_idx = df[df['is_real']==True].index

print('Sensitivity analysis — real companies flagged at each contamination rate:')
print(f'{"Company":<35} {"0.05":>6} {"0.10":>6} {"0.15":>6} {"0.20":>6}')
print('-'*60)

flags_by_rate = {}
for c in contamination_rates:
    iso_c = IsolationForest(contamination=c, n_estimators=200, random_state=42)
    preds = iso_c.fit_predict(X_scaled)
    flags_by_rate[c] = preds

for idx in real_idx:
    row     = df.loc[idx]
    results = [('🔴' if flags_by_rate[c][idx]==-1 else '⚪') for c in contamination_rates]
    name    = row['company'][:32]
    print(f'{name:<35} {results[0]:>6} {results[1]:>6} {results[2]:>6} {results[3]:>6}')

print()
print('Companies flagged at ALL four contamination rates = most robust anomalies.')
print('Companies flagged at only high rates = marginal outliers.')

---
## 8. Conclusions

### What the algorithm found — and what it means

**Finding 1: CK Mafutha is the most anomalous record in the dataset**
With the lowest anomaly score (-0.618), CK Mafutha (Pty) Ltd received R77.2M
and created just 4 jobs — R19.3M per job. This is flagged as a red flag
even ahead of Khatu Industrial (R534M, 26 jobs), because the algorithm
considers the deal unusual across ALL dimensions, not just cost-per-job.
A R77M deal creating 4 jobs is statistically more isolated in feature space
than a R534M deal creating 26 jobs.

**Finding 2: Lebowakgomo Poultry is the most anomalous positive outlier**
R1.6M disbursed, 887 jobs created — R1,804 per job. The algorithm flags this
as genuinely unusual because no other company in the dataset approaches this
efficiency at this disbursement level. This is what development finance
is supposed to look like.

**Finding 3: Red flags concentrate in Northern Cape and Western Cape**
Both provinces have above-average red flag rates driven by a small number
of very large, low-job-creation deals.

**Finding 4: Positive outliers concentrate in Limpopo, Free State, and Gauteng**
Labour-intensive sectors (agriculture, food retail, services) in these provinces
deliver the best job creation per rand — consistent with Project 3 findings.

**Finding 5: The red flags are robust across contamination rates**
Khatu Industrial, CK Mafutha, and Devland Gardens are flagged at all
contamination thresholds from 5% to 20%. These are not marginal outliers —
they are genuine anomalies by any reasonable definition of the parameter.

**Methodological note:**
Isolation Forest flags statistical anomalies — not wrongdoing. A company
flagged as a red flag may have legitimate reasons for its cost-per-job outcome
(capital-intensive infrastructure, long job creation timelines, strategic sector rationale).
The flags are a starting point for accountability questions, not conclusions.

---
*This notebook feeds into `pages/7_Anomaly_Detection.py` — a Streamlit page
that presents these findings to a public and journalistic audience.*

---

## Attribution & Authorship

**Analysis and code:** Lindiwe Songelwa

**Underlying dataset:** Compiled and made publicly available by
[@AfikaSoyamba](https://x.com/AfikaSoyamba) on X, who built a database of
1,248 South African businesses funded by the IDC and NEF — 856 from the IDC,
392 from the NEF — including every company name, amount, and province.
This analysis would not exist without that work.

**Data sources:**
- NEF: Parliamentary Question PQ705 (Mr RWC Chance, DA) via https://www.thedtic.gov.za/
- IDC: Industrial Development Corporation public funding dashboard
- Stats SA: Quarterly Labour Force Survey Q3 2025

**Part of the SA DFI Inequality Analysis series:**

| Project | Title | Status |
|---------|-------|--------|
| Project 3 | Funding Concentration & Inequality Analysis | ✅ Complete |
| Project 2 | Job Creation ROI Predictor | ✅ Complete |
| Project 5 | NEF Anomaly Detection (this notebook) | ✅ Complete |